In [ ]:
from motion import Rover, Scout, Drone
from TerrainMap import TerrainMap
from SimulationLogger import SimulationLogger
from real_time_plot import MapPlotter
from Governor import Governor, AgentState
import math
from src.map_api import MapAPI


def main(map, zig_lookahead=5.0, zig_width = 4.0, live=False):
    map_api = get_map_api("generated_maps/" + map + ".csv")
    start_pos = (10, 1)
    target = (40, 37)
    drone = Drone(map_api, "drone", start_pos)
    scout = Scout(map_api, "scout", start_pos)
    rover = Rover(map_api, "rover", start_pos)

    ten_seconds = int(1 / scout.dt * 10)
    sim_logger = SimulationLogger(log_interval=ten_seconds)
    plotter = MapPlotter(grid_size=50, live=live)
    terrain_map = TerrainMap()

    drone_state = AgentState(
        agent=drone,
        goals=[target, start_pos],
        use_zigzag=True
    )
    scout_state = AgentState(
        agent=scout,
        goals=[target, start_pos],
        use_zigzag=True,
    )
    rover_state = AgentState(
        agent=rover,
        goals=[target],
        terminal=True,
    )

    governor = Governor(terrain_map, [drone_state, scout_state, rover_state],zig_lookahead=zig_lookahead,zig_width= zig_width)

    sim_logger.start(total_steps=None,
                     start=start_pos,
                     target=target, )
    step = 0
    while True:
        step += 1
        time_elapsed = step * scout.dt

        # -- Get heading for each agent
        headings = governor.get_headings()
        if governor.done:
            reached_target = True
            break
        if time_elapsed > 100000:
            reached_target = False
            break

        # ------ Perceive ----------
        observations = {}
        observations.update(scout.perceive())
        observations.update(drone.perceive())

        # ------ Step ----------
        movement_information = {}

        if drone_state.finished:  # rover wait that drone has done a full journey
            step_rover_result = rover.step_towards(headings["rover"])
            movement_information[rover.x, rover.y] = step_rover_result

        step_scout_result = scout.step_towards(headings["scout"])
        movement_information[scout.x, scout.y] = step_scout_result
        drone.step_towards(headings["drone"])
        terrain_map.update_map(observations, movement_information)

        agents_positions = [
            (rover.x, rover.y),
            (scout.x, scout.y),
            (drone.x, drone.y)
        ]
        # 2. Package the snapshot using your method!
        snapshot = {
            "grid": terrain_map.get_grid_snapshot(),
            "agents": agents_positions
        }
        plotter.update(snapshot["grid"], governor.agents)
        sim_logger.log_step(step=step, simulation_time=time_elapsed, drone_position=(drone.x, drone.y),
                            scout_position=(scout.x, scout.y),
                            rover_position=(rover.x, rover.y))
    plotter.save(map, fps=15)

    perceive_calls = drone.method_counts["perceive"] + scout.method_counts["perceive"]
    step_calls = drone.method_counts["step"] + scout.method_counts["step"] + rover.method_counts["step"]
    stuck_event = rover.get_stuck_events_number()

    drone_travel = drone.get_travel()
    scout_travel = scout.get_travel()
    rover_travel = rover.get_travel()

    last_distance_from_target = math.sqrt((rover.x - target[0]) ** 2 + (rover.y - target[1]) ** 2)

    sim_logger.end(reached_target=reached_target,
                   last_distance_from_target=last_distance_from_target,
                   time_elapsed=time_elapsed,
                   drone_travel=drone_travel, scout_travel=scout_travel, rover_travel=rover_travel,
                   perceive_calls=perceive_calls, step_calls=step_calls,
                   stuck_calls=stuck_event,
                   )

    return reached_target, last_distance_from_target, time_elapsed, drone_travel, scout_travel, rover_travel, perceive_calls, step_calls, stuck_event
def get_map_api(csv_path):
    print("Loading MapAPI & Components...")
    map_api = MapAPI(terrain=csv_path, rng_seed=42, time_step=0.90)
    return map_api

In [ ]:
zig_lookahead = [4.5,5,5.5]
zig_width = [1.5,2.0,2.5]
numbers = [a for a in range(9)]

In [ ]:
from joblib import Parallel, delayed
from itertools import product

param_combinations = list(product(zig_lookahead, zig_width))

combinations_with_id = list(zip(param_combinations, numbers))
# Risultato: [((2.5,1.5), 0), ((2.5,2.0), 1), ((2.5,2.5), 2), ...]

def run_combination(params, number):
    rpn, rpd = params
    return main(
        map="map_013_seed13",
        revisit_penalty_scout=rpn,
        revisit_penalty_drone=rpd,
        identifier=number,  # ora corrisponde al parametro 'id' di main()
        live=False
    )

# Esegui le 9 simulazioni in parallelo
results = Parallel(n_jobs=-1, backend='loky', verbose=10)(
    delayed(run_combination)(params, number)
    for params, number in combinations_with_id
)

In [ ]:
def run_batch(n_maps: int = 10, runs_per_map: int = 3, seed: int = 42):
    """
    Cycle through n_maps maps, run each runs_per_map times with randomised
    (but always opposite) start/target positions. Print per-run scores and
    the overall average at the end.
    """
    rng  = random.Random(seed)
    maps = pick_maps(n=n_maps)

    all_results = []
    all_scores: list[float] = []
    run_counter = 0

    print("=" * 65)
    print(f"  BATCH  |  {len(maps)} maps × {runs_per_map} runs  |  seed={seed}")
    print("=" * 65)

    for map_name in maps:
        print(f"\n{'─'*65}")
        print(f"  MAP: {map_name}")
        print(f"{'─'*65}")

        for run_idx in range(runs_per_map):
            run_counter += 1
            start, target = random_opposite_positions(rng)

            print(f"\n  Run {run_counter:>3}  [{map_name}  #{run_idx+1}/{runs_per_map}]")
            print(f"         start={start}  →  target={target}")

            try:

                param_combinations = list(product(zig_lookahead, zig_width))

                combinations_with_id = list(zip(param_combinations, numbers))
                # Risultato: [((2.5,1.5), 0), ((2.5,2.0), 1), ((2.5,2.5), 2), ...]

                def run_combination(params, number):
                    rpn, rpd = params
                    return main(
                        map="map_013_seed13",
                        revisit_penalty_scout=rpn,
                        revisit_penalty_drone=rpd,
                        identifier=number,  # ora corrisponde al parametro 'id' di main()
                        live=False
                    )

                # Esegui le 9 simulazioni in parallelo
                results = Parallel(n_jobs=-1, backend='loky', verbose=10)(
                    delayed(run_combination)(params, number)
                    for params, number in combinations_with_id
                )


            except Exception as exc:
                print(f"  [ERROR] run failed: {exc}")
                # Penalise heavily so the average stays meaningful
                score = compute_score(100_000, 0, MAP_SIZE * math.sqrt(2))
                all_scores.append(score)
                print(f"  Score (failed run): {score:>12,.1f}")
                continue
            all_results.append(results)

In [ ]:
            score = compute_score(T, n_stuck, dist)
            all_scores.append(score)

            # ── Per-run output ───────────────────────────────────────────────
            print(f"  {'Success':<24} {'YES ✓' if reached else 'NO  ✗'}")
            print(f"  {'Mission time  T_i':<24} {T:>12,.1f}  s")
            print(f"  {'Stuck events  N_i':<24} {n_stuck:>12}")
            print(f"  {'Final distance  D_i':<24} {dist:>12.2f}  cells")
            print(f"  {'λ_s · N_i':<24} {LAMBDA_STUCK * n_stuck:>12,.0f}")
            print(f"  {'λ_d · D_i':<24} {LAMBDA_DISTANCE * dist:>12,.1f}")
            print(f"  {'·'*40}")
            print(f"  {'Score  J_i':<24} {score:>12,.1f}")

In [ ]:
MAP_SIZE        = 50       # grid is 50x50
POSITION_MARGIN = 5        # min distance from border for start/target
import random


def random_opposite_positions(rng: random.Random, map_size: int = MAP_SIZE, margin: int = POSITION_MARGIN):
    """
    Sample start and target so they always lie in opposite halves of the map.

    Start  → sampled from [margin, half - margin]  on both axes
    Target → sampled from [half + margin, size - margin] on both axes

    This guarantees the two points are always on opposite sides regardless of
    which specific values are drawn.
    """
    half = map_size // 2
    sx = rng.randint(margin, half - margin)
    sy = rng.randint(margin, half - margin)
    tx = rng.randint(half + margin, map_size - margin)
    ty = rng.randint(half + margin, map_size - margin)
    return (sx, sy), (tx, ty)


In [ ]:
# ── Scoring constants ────────────────────────────────────────────────────────
LAMBDA_STUCK    = 50_000   # penalty per stuck event
LAMBDA_DISTANCE = 7_000    # penalty per cell of final distance to target

def compute_score(time_elapsed: float, n_stuck: int, dist_final: float) -> float:
    """J_i = T_i + λ_s · N_stuck + λ_d · D_final"""
    return time_elapsed + LAMBDA_STUCK * n_stuck + LAMBDA_DISTANCE * dist_final


In [ ]:
import os
def pick_maps(folder: str = "generated_maps", n: int = 10) -> list[str]:
    """Return up to n map names (no extension) found in folder, sorted."""
    files = sorted(
        f[:-4] for f in os.listdir(folder) if f.endswith(".csv")
    )
    if len(files) < n:
        print(f"[WARN] Only {len(files)} maps found in '{folder}', using all of them.")
    return files[:n]


def cell_distance(pos, target) -> float:
    """Euclidean distance in cell units."""
    return math.sqrt((pos[0] - target[0]) ** 2 + (pos[1] - target[1]) ** 2)




In [ ]:
def get_map_api(csv_path: str) -> MapAPI:
    print(f"  Loading map: {csv_path}")
    return MapAPI(terrain=csv_path, rng_seed=42, time_step=0.90)